# SpatialVista：小鼠脑数据示例

这个 notebook 用于检查并可视化 `mouse_brain_3_IQ.h5ad`。请从上到下运行单元格。该文件包含处理后的空间坐标、注释和性状分数，但不包含完整的基因表达矩阵。

In [1]:
from pathlib import Path
import anndata as ad
import spatialvista as spv

DATA_PATH = Path(r"D:\test\data\mouse_brain_3_IQ.h5ad")
assert DATA_PATH.exists(), f"找不到数据文件：{DATA_PATH}"
DATA_PATH

WindowsPath('D:/test/data/mouse_brain_3_IQ.h5ad')

## 1. 不把全部数据读进内存，先检查文件结构

`backed="r"` 会直接从硬盘读取，适合先检查大型 AnnData 文件，避免立刻占用大量内存。

In [3]:
adata_backed = ad.read_h5ad(DATA_PATH, backed="r")
print(adata_backed)
print("obs 元数据列：", list(adata_backed.obs.columns))
print("obsm 坐标矩阵：", {k: adata_backed.obsm[k].shape for k in adata_backed.obsm.keys()})
print("变量：", list(adata_backed.var_names))
adata_backed.obs.head()

AnnData object with n_obs × n_vars = 2055846 × 1 backed at 'D:\\test\\data\\mouse_brain_3_IQ.h5ad'
    obs: '3d_x', '3d_y', '3d_z', 'sx', 'sy', 'sample_name', 'Intelligence', 'Puberty_time_2025_Nature_Genetics', 'cell_type', 'major_brain_region'
    obsm: 'spatial', 'spatial_2d', 'spatial_3d'
    layers: None (.X)
obs columns: ['3d_x', '3d_y', '3d_z', 'sx', 'sy', 'sample_name', 'Intelligence', 'Puberty_time_2025_Nature_Genetics', 'cell_type', 'major_brain_region']
obsm coordinates: {'spatial': (2055846, 3), 'spatial_2d': (2055846, 2), 'spatial_3d': (2055846, 3)}
variables: ['0']


,3d_x,3d_y,3d_z,sx,sy,sample_name,Intelligence,Puberty_time_2025_Nature_Genetics,cell_type,major_brain_region
spot,,,,,,,,,,
1000002106742124543660891200204200272|WB_imputation_animal3_sagittal_C57BL6J-3.005,811.468950,3633.071100,4716.389725,811.468950,3633.071100,WB_imputation_animal3_sagittal_C57BL6J-3.005,8.686771,2.213670,GABAergic neuron,Olfactory
100000305306995484992604798000963664554|WB_imputation_animal3_sagittal_C57BL6J-3.006,12070.503450,3122.478900,4555.166650,12070.503450,3122.478900,WB_imputation_animal3_sagittal_C57BL6J-3.006,5.106364,1.621589,astrocyte,Cerebellum
100000409474216812404625224199719978987|WB_imputation_animal3_sagittal_C57BL6J-3.002,11561.861500,5197.586350,5321.370500,11561.861500,5197.586350,WB_imputation_animal3_sagittal_C57BL6J-3.002,5.168264,2.104278,astrocyte,Medulla
10000084327746466562688083123631179221|WB_imputation_animal3_sagittal_C57BL6J-3.008,8686.343775,4784.102825,4073.617075,8686.343775,4784.102825,WB_imputation_animal3_sagittal_C57BL6J-3.008,5.622371,2.389561,astrocyte,Midbrain
100000950651718129729129473767119181938|WB_imputation_animal3_sagittal_C57BL6J-3.010,6441.564600,5080.579925,3682.646300,6441.564600,5080.579925,WB_imputation_animal3_sagittal_C57BL6J-3.010,11.753052,5.150995,GABAergic neuron,Pallidum


In [4]:
summary = {
    "点或细胞数": adata_backed.n_obs,
    "切片数": adata_backed.obs["sample_name"].nunique(),
    "细胞类型数": adata_backed.obs["cell_type"].nunique(),
    "主要脑区数": adata_backed.obs["major_brain_region"].nunique(),
}
summary

{'spots': 2055846, 'samples': 22, 'cell_types': 22, 'brain_regions': 14}

In [5]:
adata_backed.obs["major_brain_region"].value_counts().to_frame("点或细胞数")

,spots
major_brain_region,
Isocortex,441589
Cerebellum,306499
Olfactory,270461
Fiber_tracts,178963
Striatum,169661
Hippocampus,151817
Midbrain,138405
Medulla,104258
Thalamus,68501


In [6]:
adata_backed.file.close()

## 2. 将数据读入内存并打开交互式三维视图

下一个单元格会把处理后数据完整读入内存。该对象包含 2,055,846 个空间点，因此可能需要等待一会儿。

In [7]:
adata = ad.read_h5ad(DATA_PATH)
adata

AnnData object with n_obs × n_vars = 2055846 × 1
    obs: '3d_x', '3d_y', '3d_z', 'sx', 'sy', 'sample_name', 'Intelligence', 'Puberty_time_2025_Nature_Genetics', 'cell_type', 'major_brain_region'
    obsm: 'spatial', 'spatial_2d', 'spatial_3d'
    layers: None (.X)

In [8]:
viewer = spv.vis(
    adata,
    position="spatial_3d",
    color="major_brain_region",
    section="sample_name",
    annotations=["cell_type", "sample_name"],
    continuous=["Intelligence", "Puberty_time_2025_Nature_Genetics"],
    height=800,
)
viewer